# c10：标题预切分 + 方案 D — 从 OSS 到 Elasticsearch

本笔记本演示 **Markdown ATX 标题递归预切分**（`leaf_ranges_heading_presplit`）后，对超长叶子在段内运行 **方案 D**（`document_segmentation`），再经 **BGE 向量化** 写入 **ES** 的完整链路。

与正式脚本 [`scripts/ingest_oss_md_to_es.py`](../scripts/ingest_oss_md_to_es.py)（整篇滑窗 + 可选语义合并）不同，此处与 [`scripts/d05_heading_presplit_document_segmentation_export.py`](../scripts/d05_heading_presplit_document_segmentation_export.py) 及 [`chunking.md_heading_presplit`](../src/chunking/md_heading_presplit.py) 对齐。

每条 chunk 的 **`document_chunk_id`**（文档内序号）、**`section_heading_id`**（标题锚点，形如 `source_path#h{level}@{char}` 或 `source_path#prelude@{leaf_start}`）、**`heading_level`**（可选）会进入 ES `_source` 顶层，便于过滤与对账；叶子边界等调试信息保留在 **`extra`** 中。

**批量 OSS**：默认 **`C10_OSS_SCOPE=all`**，对 `OSS_BUCKET` + `OSS_OBJECT_PREFIX` 下列举到的全部 `.md` 下载、切分、向量化并写入 ES；烟测可设 **`C10_OSS_SCOPE=filter`**，仅处理 key 中含 **`C10_NAME_FILTER`**（默认「婚姻法」）的对象。

## 依赖与环境变量

在项目根执行（一次）：

```bash
uv sync --extra oss --extra segmentation --extra embedding --extra es
```

说明：默认 `uv sync` 不含 OSS SDK；未装 **`oss`** 组时列举/下载会报 `No module named 'alibabacloud_oss_v2'`，请确保命令中含 **`--extra oss`**。

| 用途 | 环境变量 / Settings 字段 |
|------|-------------------------|
| OSS 访问 | `OSS_ACCESS_KEY_ID`, `OSS_ACCESS_KEY_SECRET`, `OSS_REGION`, 可选 `OSS_ENDPOINT` |
| 桶与前缀 | `OSS_BUCKET`, `OSS_OBJECT_PREFIX`；下载目录 `OSS_DOWNLOAD_DIR` |
| 方案 D 模型目录 | `DOCUMENT_SEGMENTATION_PATH` |
| 段内 D 参数 | `DOC_SEGMENTATION_MIN_CHARS`, `DOC_SEGMENTATION_MAX_CHARS`, `DOC_SEGMENTATION_SPLIT_OVERLAP`, `DOC_SEGMENTATION_SECTION_MAX_CHARS` |
| 标题策略 | `CHUNK_MD_HEADING_STRATEGY`, `CHUNK_MD_HEADING_FIXED_LEVEL`, `CHUNK_MD_HEADING_SINGLE_IMMEDIATE_CHILD` |
| 向量模型 | `BGE_M3_PATH` 等（与 `embeddings` 一致） |
| ES | `ES_HOST`, `ES_PORT`, `ES_INDEX`；本笔记本建议另设 **`ES_INDEX_C10`**（见下一节），避免覆盖生产索引 |

可选：在 shell 中 `export ES_INDEX_C10=rag_law_c10_nb`，或在下方代码单元设置。

| 笔记本专用 | 含义 |
|------------|------|
| `C10_OSS_SCOPE` | `all`（默认）：前缀下全部 `.md`；`filter`：仅 key 含 `C10_NAME_FILTER` |
| `C10_NAME_FILTER` | 与 `filter` 联用，默认 `婚姻法` |
| `C10_EMBED_BATCH` | 向量化批大小，默认 `32` |
| `C10_ES_BULK_BATCH` | ES bulk 每批条数，默认 `200` |
| `C10_OSS_MAX_FILES` | 可选：仅处理字典序前 N 个 `.md`（全量调试时防跑太久） |
| `C10_PREVIEW_CHUNKS` | 抽样打印 chunk 条数上限，默认 `40` |
| `LOCAL_MD_FALLBACK` | 设后跳过 OSS，只处理单个本地 `.md` |

In [ ]:
import os
import sys
from pathlib import Path

# 以「项目根」为 cwd（在 notebooks/ 下运行时会自动上溯）
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(ROOT)
print("ROOT =", ROOT)

# 使用独立 ES 索引名，避免误删生产数据；也可事先 export ES_INDEX_C10=...
os.environ.setdefault("ES_INDEX_C10", "rag_law_c10_nb")
os.environ["ES_INDEX"] = os.environ["ES_INDEX_C10"]

ROOT = /Users/zheng/projects/python/ai/rag/rag_law


In [ ]:
from conf.settings import get_settings

get_settings.cache_clear()
st = get_settings()
rows = [
    ("ES_INDEX", st.es_index),
    ("OSS_BUCKET", st.oss_bucket),
    ("OSS_OBJECT_PREFIX", st.oss_object_prefix),
    ("OSS_DOWNLOAD_DIR", str(st.oss_download_dir)),
    ("DOCUMENT_SEGMENTATION_PATH", st.document_segmentation_path or ""),
    ("DOC_SEGMENTATION_*", f"min={st.doc_segmentation_min_chars} max={st.doc_segmentation_max_chars} ov={st.doc_segmentation_split_overlap} section_max={st.doc_segmentation_section_max_chars or st.doc_segmentation_max_chars}"),
    ("CHUNK_MD_HEADING_*", f"strategy={st.chunk_md_heading_strategy} fixed_level={st.chunk_md_heading_fixed_level} single_child={st.chunk_md_heading_single_immediate_child}"),
]
for k, v in rows:
    print(f"{k}: {v}")

ES_INDEX: rag_law_c10_nb
OSS_BUCKET: rag-law
OSS_OBJECT_PREFIX: md3/
OSS_DOWNLOAD_DIR: data/md_minerU
DOCUMENT_SEGMENTATION_PATH: /Users/zheng/projects/models/nlp/bert/nlp_bert_document-segmentation_chinese-base
DOC_SEGMENTATION_*: min=200 max=800 ov=0 section_max=800
CHUNK_MD_HEADING_*: strategy=deepest_with_multiple fixed_level=None single_child=strict


## 1. 获取 Markdown（OSS：全量或过滤）

与 [`scripts/ingest_oss_md_to_es.py`](../scripts/ingest_oss_md_to_es.py) 一致：列举 `OSS_BUCKET` + `OSS_OBJECT_PREFIX` 下全部 `.md`，下载到 `OSS_DOWNLOAD_DIR`，并用 [`ingest.oss_layout.iter_oss_md_ingest_plan`](../src/ingest/oss_layout.py) 生成与入库一致的 `source_path` / 本地路径。

- **默认全量**：`C10_OSS_SCOPE=all`（或不设），处理前缀下**每一个** `.md`。
- **烟测过滤**：`export C10_OSS_SCOPE=filter` 时仅下载 key 中含 **`C10_NAME_FILTER`** 的文件（默认子串「婚姻法」）。

若未配置 OSS，可设 **`LOCAL_MD_FALLBACK`** 指向单个本地 `.md`，此时不走 OSS 列举。

走 OSS 须安装 **`alibabacloud-oss-v2`**（`uv sync --extra oss`）并重启 Jupyter 内核。

In [ ]:
import importlib.util
from pathlib import Path

from conf.settings import get_settings
from ingest.oss_layout import iter_oss_md_ingest_plan
from utils.oss_aliyun.public_urls import oss_virtual_host_public_url


def _oss_sdk_available() -> bool:
    return importlib.util.find_spec("alibabacloud_oss_v2") is not None


get_settings.cache_clear()
st = get_settings()
bucket = st.oss_bucket
prefix = (st.oss_object_prefix or "").strip()
if prefix and not prefix.endswith("/"):
    prefix += "/"

download_dir = Path(st.oss_download_dir)
if not download_dir.is_absolute():
    download_dir = (ROOT / download_dir).resolve()
download_dir.mkdir(parents=True, exist_ok=True)

C10_OSS_SCOPE = os.environ.get("C10_OSS_SCOPE", "all").strip().lower()
C10_NAME_FILTER = os.environ.get("C10_NAME_FILTER", "婚姻法").strip()
local_fallback = os.environ.get("LOCAL_MD_FALLBACK", "").strip()

url_by_source_path: dict[str, str] = {}
# (source_path, local_path, oss_key)；本地兜底时 oss_key 为空
md_plans: list[tuple[str, Path, str]] = []

if local_fallback:
    lp = Path(local_fallback).expanduser()
    if not lp.is_absolute():
        lp = (ROOT / lp).resolve()
    try:
        sp = lp.resolve().relative_to(ROOT.resolve()).as_posix()
    except ValueError:
        sp = str(lp.resolve())
    md_plans = [(sp, lp, "")]
    print("LOCAL_MD_FALLBACK:", lp)
else:
    if not _oss_sdk_available():
        raise RuntimeError(
            "未检测到 alibabacloud_oss_v2（OSS 可选依赖）。请在项目根执行：\n"
            "  uv sync --extra oss\n"
            "然后重启 Jupyter 内核；或设置 LOCAL_MD_FALLBACK=/path/to/file.md。\n"
            "（完整：uv sync --extra oss --extra segmentation --extra embedding --extra es）"
        )
    from utils.oss_aliyun.crud import download_object_to_file, list_all_object_keys

    region = (st.oss_region or os.getenv("OSS_REGION") or "").strip()
    if not region:
        raise RuntimeError("未设置 OSS_REGION，或设置 LOCAL_MD_FALLBACK")
    keys = [k for k in sorted(list_all_object_keys(bucket, prefix=prefix)) if k.lower().endswith(".md")]
    if C10_OSS_SCOPE == "filter":
        if not C10_NAME_FILTER:
            raise RuntimeError("C10_OSS_SCOPE=filter 时需设置 C10_NAME_FILTER")
        keys = [k for k in keys if C10_NAME_FILTER in k]
        print(f"过滤模式: key 含「{C10_NAME_FILTER}」→ {len(keys)} 个 .md")
    else:
        print(f"全量模式: 前缀下共 {len(keys)} 个 .md")
    if not keys:
        raise RuntimeError("无可用 .md，请检查 OSS_OBJECT_PREFIX 或过滤条件")
    _maxf = os.environ.get("C10_OSS_MAX_FILES", "").strip()
    if _maxf.isdigit() and int(_maxf) > 0:
        keys = keys[: int(_maxf)]
        print(f"C10_OSS_MAX_FILES={_maxf}：仅处理前 {len(keys)} 个 key")
    md_plans = iter_oss_md_ingest_plan(ROOT, download_dir, keys)
    for sp, lp, key in md_plans:
        download_object_to_file(bucket, key, lp)
        url_by_source_path[sp] = oss_virtual_host_public_url(bucket, region, key)

print("待处理文件数:", len(md_plans))
for sp, lp, ok in md_plans[:8]:
    print(" ", sp, "|", lp.name)
if len(md_plans) > 8:
    print(f" ... 共 {len(md_plans)} 个文件（上列为前 8）")

全量模式: 前缀下共 8 个 .md
待处理文件数: 8
  data/md_minerU/md3__刑法.md | md3__刑法.md
  data/md_minerU/md3__劳动合同法.md | md3__劳动合同法.md
  data/md_minerU/md3__劳动法.md | md3__劳动法.md
  data/md_minerU/md3__婚姻法.md | md3__婚姻法.md
  data/md_minerU/md3__宪法.md | md3__宪法.md
  data/md_minerU/md3__民事诉讼法.md | md3__民事诉讼法.md
  data/md_minerU/md3__民法典.md | md3__民法典.md
  data/md_minerU/md3__行政复议法.md | md3__行政复议法.md


## 2. 标题预切分 + 段内方案 D（多文件循环）

对 **`md_plans`** 中每个文件读取全文，共用**一个** `document_segmentation` pipeline，依次产出 `TextChunk` 并累积到 **`all_chunks`**（每个 chunk 的 `source_file` / `source_path` 已区分来源）。

1. `leaf_ranges_heading_presplit`：每文件独立得到叶子区间。  
2. `iter_heading_presplit_document_segmentation_chunks_for_text`：短叶整块、长叶段内方案 D。  
3. `extra` 中的 `document_chunk_id` 等按**单文件内**从 0 递增；入库时用各文件 SHA256 与 `source_oss_url` 对齐。

In [ ]:
from chunking.document_segmentation import build_document_segmentation_pipeline
from chunking.md_heading_presplit import (
    leaf_ranges_heading_presplit,
    iter_heading_presplit_document_segmentation_chunks_for_text,
)

st = get_settings()
model_dir = Path(st.document_segmentation_path or "").expanduser().resolve()
if not model_dir.is_dir():
    raise RuntimeError("请在 .env 配置 DOCUMENT_SEGMENTATION_PATH 指向有效模型目录")

pipe = build_document_segmentation_pipeline(str(model_dir))
section_max = st.doc_segmentation_section_max_chars or st.doc_segmentation_max_chars

all_chunks: list = []
per_file_stats: list[tuple[str, int, int]] = []  # source_path, 叶子数, chunk数

for sp, local_p, _key in md_plans:
    text_one = local_p.read_text(encoding="utf-8")
    sf = local_p.name
    leaves = leaf_ranges_heading_presplit(
        text_one,
        strategy=st.chunk_md_heading_strategy,  # type: ignore[arg-type]
        fixed_first_level=st.chunk_md_heading_fixed_level,
        single_immediate_child=st.chunk_md_heading_single_immediate_child,  # type: ignore[arg-type]
    )
    file_chunks = list(
        iter_heading_presplit_document_segmentation_chunks_for_text(
            text_one,
            source_file=sf,
            source_path=sp,
            pipeline=pipe,
            leaf_ranges=leaves,
            min_chars=st.doc_segmentation_min_chars,
            max_chars=st.doc_segmentation_max_chars,
            split_overlap=st.doc_segmentation_split_overlap,
            section_max_chars=section_max,
        )
    )
    per_file_stats.append((sp, len(leaves), len(file_chunks)))
    all_chunks.extend(file_chunks)

print("文件数:", len(md_plans), "总 chunk 数:", len(all_chunks))
print("各文件 (source_path, 叶子数, chunk数) 前 15 行:")
for row in per_file_stats[:15]:
    print(" ", row)
if len(per_file_stats) > 15:
    print(f" ... 共 {len(per_file_stats)} 个文件")

2026-04-12 22:49:42,077 - modelscope - INFO - initiate model from /Users/zheng/projects/models/nlp/bert/nlp_bert_document-segmentation_chinese-base
2026-04-12 22:49:42,077 - modelscope - INFO - initiate model from location /Users/zheng/projects/models/nlp/bert/nlp_bert_document-segmentation_chinese-base.
2026-04-12 22:49:42,079 - modelscope - INFO - initialize model from /Users/zheng/projects/models/nlp/bert/nlp_bert_document-segmentation_chinese-base
2026-04-12 22:49:42,145 - modelscope - WARNING - No preprocessor field found in cfg.
2026-04-12 22:49:42,146 - modelscope - WARNING - No val key and type key found in preprocessor domain of configuration.json file.
2026-04-12 22:49:42,146 - modelscope - WARNING - Cannot find available config to build preprocessor at mode inference, current config: {'model_dir': '/Users/zheng/projects/models/nlp/bert/nlp_bert_document-segmentation_chinese-base'}. trying to build by task and model information.
2026-04-12 22:49:42,146 - modelscope - INFO - N

文件数: 8 总 chunk 数: 629
各文件 (source_path, 叶子数, chunk数) 前 15 行:
  ('data/md_minerU/md3__刑法.md', 59, 136)
  ('data/md_minerU/md3__劳动合同法.md', 13, 23)
  ('data/md_minerU/md3__劳动法.md', 15, 19)
  ('data/md_minerU/md3__婚姻法.md', 8, 10)
  ('data/md_minerU/md3__宪法.md', 14, 31)
  ('data/md_minerU/md3__民事诉讼法.md', 52, 79)
  ('data/md_minerU/md3__民法典.md', 175, 301)
  ('data/md_minerU/md3__行政复议法.md', 23, 30)


## 3. 抽样打印切分结果

默认只打印 **`all_chunks`** 的前若干条（含 `source_file`）。全量入库前建议先 **`C10_OSS_SCOPE=filter`** 跑通，再改 **`all`** 跑全桶。

In [ ]:
PREVIEW = 380
PRINT_FIRST = int(os.environ.get("C10_PREVIEW_CHUNKS", "40"))
for c in all_chunks[:PRINT_FIRST]:
    ex = c.extra or {}
    dcid = ex.get("document_chunk_id", c.chunk_index)
    sid = ex.get("section_heading_id", "")
    hl = ex.get("heading_level", "")
    sd = ex.get("scheme_d", False)
    ck = ex.get("chunking", "")
    snippet = (c.text[:PREVIEW] + "…") if len(c.text) > PREVIEW else c.text
    print("---")
    print(
        f"src={c.source_file!r} document_chunk_id={dcid} chunk_index={c.chunk_index} "
        f"[{c.char_start},{c.char_end}) scheme_d={sd} chunking={ck}"
    )
    print(f"section_heading_id={sid!r} heading_level={hl!r}")
    print(snippet.replace("\n", "\\n"))
if len(all_chunks) > PRINT_FIRST:
    print(f"... 共 {len(all_chunks)} 条，此处仅展示前 {PRINT_FIRST} 条")

if all_chunks:
    print("第一条 extra keys:", sorted((all_chunks[0].extra or {}).keys()))

---
src='md3__刑法.md' document_chunk_id=0 chunk_index=0 [0,678) scheme_d=False chunking=heading_presplit_leaf
section_heading_id='data/md_minerU/md3__刑法.md#h1@0' heading_level=1
# 中华人民共和国刑法\n\n第一编 总则\n\n第一章 刑法的任务、基本原则和适用范围\n\n第二章 犯罪\n\n第一节 犯罪和刑事责任\n\n第二节 犯罪的预备、未遂和中止\n\n第三节 共同犯罪\n\n第四节 单位犯罪\n\n第三章 刑罚\n\n第一节 刑罚的种类\n\n第二节 管制\n\n第三节 拘役\n\n第四节 有期徒刑、无期徒刑\n\n第五节 死刑\n\n第六节 罚金\n\n第七节 剥夺政治权利\n\n第八节 没收财产\n\n第四章 刑罚的具体运用\n\n第一节 量刑\n\n第二节 累犯\n\n第三节 自首和立功\n\n第四节 数罪并罚\n\n第五节 缓刑\n\n第六节 减刑\n\n第七节 假释\n\n第八节 时效\n\n第五章 其他规定\n\n第二编 分则\n\n第一章 危害国家安全罪\n\n第二章 危害公共安全罪\n\n第三章 破坏社会主义市场经济秩序罪\n\n第一节 生产、销售伪劣商品罪\n\n第二节 走私罪\n\n第三节 妨害对公司、企业的管理秩序罪\n\n第四…
---
src='md3__刑法.md' document_chunk_id=1 chunk_index=1 [678,684) scheme_d=False chunking=heading_presplit_leaf
section_heading_id='data/md_minerU/md3__刑法.md#h1@678' heading_level=1
# 条文\n\n
---
src='md3__刑法.md' document_chunk_id=2 chunk_index=2 [684,694) scheme_d=False chunking=heading_presplit_leaf
section_heading_id='data/md_minerU/md3__刑法.md#h1@684' heading_level=1
# 

## 4. 向量化、写入 ES、抽检

- **`recreate=True`** 会删除当前 `ES_INDEX`（本笔记本默认为 `rag_law_c10_nb`）后重建；**请勿对生产索引名运行**。  
- 向量化按 **`C10_EMBED_BATCH`**（默认 32）分批调用 `embed_documents`，避免一次塞满显存/内存。  
- 写入按 **`C10_ES_BULK_BATCH`**（默认 200）分批 `bulk_index_chunks`。  
- 每条文档的 `source_sha256` / `source_oss_url` 按 **`source_path`** 从 `md_plans` 对应文件对齐。

In [ ]:
from embeddings import build_embedder
from es_store.client import elasticsearch_client
from es_store.store import EsChunkStore
from ingest.documents import chunk_embedding_to_source, sha256_utf8_file

st = get_settings()
sha_by_path = {sp: sha256_utf8_file(lp) for sp, lp, _ in md_plans}

EMBED_BATCH = max(1, int(os.environ.get("C10_EMBED_BATCH", "32")))
ES_BULK_BATCH = max(1, int(os.environ.get("C10_ES_BULK_BATCH", "200")))

embedder = build_embedder(st)
dim = embedder.dense_dimension

all_vecs: list = []
for i in range(0, len(all_chunks), EMBED_BATCH):
    batch = all_chunks[i : i + EMBED_BATCH]
    all_vecs.extend(embedder.embed_documents([c.text for c in batch]))
    done = min(i + EMBED_BATCH, len(all_chunks))
    step = max(EMBED_BATCH * 25, 500)
    if done == len(all_chunks) or done % step == 0:
        print(f"  已向量化 {done}/{len(all_chunks)}")

assert len(all_vecs) == len(all_chunks)

docs = []
for c, emb in zip(all_chunks, all_vecs):
    spk = c.source_path or ""
    docs.append(
        chunk_embedding_to_source(
            c,
            emb,
            source_sha256=sha_by_path.get(spk, ""),
            source_oss_url=url_by_source_path.get(spk, ""),
            chunk_version=st.chunk_version,
        )
    )

client = elasticsearch_client(st)
if not client.ping():
    raise RuntimeError("Elasticsearch ping 失败")

store = EsChunkStore(client, st.es_index, dense_dims=dim)
store.ensure_index(recreate=True)

ok_total = 0
for j in range(0, len(docs), ES_BULK_BATCH):
    sub = docs[j : j + ES_BULK_BATCH]
    ok, errs = store.bulk_index_chunks(sub)
    if errs:
        raise RuntimeError(errs)
    ok_total += ok
    print(f"  ES bulk 已写入 {min(j + ES_BULK_BATCH, len(docs))}/{len(docs)}")

print("bulk 成功:", ok_total, "index=", st.es_index, "dim=", dim)
store.refresh()
print("count:", client.count(index=st.es_index)["count"])

if docs:
    src0 = docs[0]
    print(
        "首条顶层: document_chunk_id=%r section_heading_id=%r heading_level=%r chunk_index=%r source_file=%r"
        % (
            src0.get("document_chunk_id"),
            src0.get("section_heading_id"),
            src0.get("heading_level"),
            src0.get("chunk_index"),
            src0.get("source_file"),
        )
    )

pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 174.68it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 1/1 [00:09<00:00,  9.66s/it]


  已向量化 629/629
  ES bulk 已写入 200/629
  ES bulk 已写入 400/629
  ES bulk 已写入 600/629
  ES bulk 已写入 629/629
bulk 成功: 629 index= rag_law_c10_nb dim= 1024
count: 629
首条顶层: document_chunk_id=0 section_heading_id='data/md_minerU/md3__刑法.md#h1@0' heading_level=1 chunk_index=0 source_file='md3__刑法.md'


In [ ]:
# 5) 读后检索：match 正文 + kNN 示例

q_text = "婚姻法 离婚"
resp = client.search(
    index=st.es_index,
    size=3,
    query={"match": {"text": {"query": q_text}}},
    _source=["text", "section_heading_id", "document_chunk_id", "chunk_index", "source_file"],
)
for h in resp["hits"]["hits"]:
    s = h["_source"]
    prev = (s.get("text") or "")[:160].replace("\n", " ")
    print("score=", h.get("_score"), "section_heading_id=", s.get("section_heading_id"), "|", prev, "…")

qv = embedder.embed_query(q_text)
hits = store.search_knn(qv, k=min(5, len(all_chunks)))
for h in hits[:3]:
    src = h.get("source") or {}
    print("kNN score=%.4f id=%r" % (h["score"], h.get("id")), (src.get("text") or "")[:120].replace("\n", " "), "…")

score= 22.70941 section_heading_id= data/md_minerU/md3__民法典.md#h1@112719 | # 第四章 离 婚  第一千零七十六条 【协议离婚】夫妻双方自愿离婚的,应当签订书面离婚协议,并亲自到婚姻登记机关申请离婚登记。  离婚协议应当载明双方自愿离婚的意思表示和对子女抚养、财产以及债务处理等事项协商一致的意见。  第一千零七十七条 【离婚冷静期】自婚姻登记机关收到离婚登记申请之日起三十日内,任何一方不愿意离 …
score= 21.411572 section_heading_id= data/md_minerU/md3__民法典.md#h1@145077 | 3.关于家庭关系。第五编第三章规定了夫妻关系、父母子女关系和其他近亲属关系,并根据社会发展需要,在现行婚姻法的基础上,完善了有关内容:一是明确了夫妻共同债务的范围。现行婚姻法没有对夫妻共同债务的范围作出规定。2003年最高人民法院出台司法解释,对夫妻共同债务的认定作出规定,近年来成为社会关注的热点问题。2018年1 月 …
score= 21.37272 section_heading_id= data/md_minerU/md3__婚姻法.md#h1@2434 | # 第四章 离婚  第三十一条 男女双方自愿离婚的，准予离婚。双方必须到婚姻登记机关申请离婚。婚姻登记机关查明双方确实是自愿并对子女和财产问题已有适当处理时，发给离婚证。  第三十二条 男女一方要求离婚的，可由有关部门进行调解或直接向人民法院提出离婚诉讼。  人民法院审理离婚案件，应当进行调解；如感情确已破裂，调解无效 …


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  4.72it/s]

kNN score=0.8721 id='md3__民法典.md:227' # 第四章 离 婚  第一千零七十六条 【协议离婚】夫妻双方自愿离婚的,应当签订书面离婚协议,并亲自到婚姻登记机关申请离婚登记。  离婚协议应当载明双方自愿离婚的意思表示和对子女抚养、财产以及债务处理等事项协商一致的意见。  第一千零七十七 …
kNN score=0.8663 id='md3__婚姻法.md:6' # 第四章 离婚  第三十一条 男女双方自愿离婚的，准予离婚。双方必须到婚姻登记机关申请离婚。婚姻登记机关查明双方确实是自愿并对子女和财产问题已有适当处理时，发给离婚证。  第三十二条 男女一方要求离婚的，可由有关部门进行调解或直接向人民法 …
kNN score=0.8475 id='md3__民法典.md:229' 第一千零八十九条 【离婚时夫妻共同债务清偿】离婚时,夫妻共同债务应当共同偿还。共同财产不足清偿或者财产归各自所有的,由双方协议清偿;协议不成的,由人民法院判决。  第一千零九十条 【离婚经济帮助】离婚时,如果一方生活困难,有负担能力的另一方 …
